# Option Portfolio Metrics

Loads **your specific option positions** from `data/option_data.csv`, enriches them with
**fresh implied volatility** interpolated from the downloaded market chain (SQLite DB),
then computes Greeks, probabilities and payoff for those positions only.

The full downloaded chain is used purely as an IV surface reference — it is never directly analysed.

## 1 · Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

from skew import (
    compute_option_metrics,
    portfolio_greeks,
    market_implied_pdf,
    prob_profit_from_pdf,
    load_option_snapshots,
)

In [ ]:
# Auto-detect project root
_cwd = Path(os.getcwd())
if   (_cwd / "skew").exists():        _project_root = _cwd
elif (_cwd.parent / "skew").exists(): _project_root = _cwd.parent
else:                                  _project_root = _cwd

PORTFOLIO_CSV = str(_project_root / "data" / "option_data.csv")
DB_PATH       = str(_project_root / "data" / "options.db")

# None = use the most recent snapshot in the DB for IV interpolation
SNAPSHOT_DATE = None
DEFAULT_RF    = 0.04
CONTRACT_SIZE = 100     # shares per contract

print(f"Portfolio : {PORTFOLIO_CSV}")
print(f"Market DB : {DB_PATH}")

In [ ]:
## 2 · Load portfolio positions

pf_raw = pd.read_csv(PORTFOLIO_CSV)

# ── Normalise columns ──────────────────────────────────────────────────────────
pf = pf_raw.copy()

# option_type → "C" / "P"
if "option_type" in pf.columns:
    pf["option_type"] = (
        pf["option_type"].str.lower().str.strip()
        .map({"call": "C", "put": "P", "c": "C", "p": "P"})
        .fillna(pf["option_type"].str.upper().str.strip())
    )
elif "isCall" in pf.columns:
    pf["option_type"] = pf["isCall"].map({True: "C", False: "P"})

# expiry as datetime
pf["expiry"] = pd.to_datetime(pf["expiry"], dayfirst=False, errors="coerce")

# T (years to expiry from today)
pf["T"] = (pf["expiry"] - pd.Timestamp.today().normalize()).dt.days / 365.0
pf["T"] = pf["T"].clip(lower=1e-6)

# spot / underlying_price
if "underlying_price" in pf.columns and "spot" not in pf.columns:
    pf = pf.rename(columns={"underlying_price": "spot"})

# premium column
if "option_price" not in pf.columns and "lastPrice" in pf.columns:
    pf = pf.rename(columns={"lastPrice": "option_price"})

# contracts default
if "contracts" not in pf.columns:
    pf["contracts"] = 1

# risk-free rate default
if "risk_free_rate" not in pf.columns:
    pf["risk_free_rate"] = DEFAULT_RF

# dividend yield default
if "div_yield" not in pf.columns:
    pf["div_yield"] = 0.0

print(f"Loaded {len(pf)} portfolio positions  |  tickers: {sorted(pf['symbol'].unique())}")
pf[["symbol","option_type","strike","expiry","T","option_price","contracts"]].head(10)

In [ ]:
## 3 · Interpolate current market IV for each position

def _load_chain(ticker, db_path, snapshot_date):
    """Return the latest (or specific) market chain for a ticker from the DB."""
    raw = load_option_snapshots(ticker, db_path=db_path, snapshot_date=snapshot_date)
    if raw.empty:
        return pd.DataFrame()
    snap = snapshot_date or raw["snapshot_date"].max()
    df = raw[raw["snapshot_date"] == snap].copy()
    df["expiry"] = pd.to_datetime(df["expiry"])
    # normalise option_type to C/P
    df["option_type"] = (
        df["option_type"].str.lower().str.strip()
        .map({"call": "C", "put": "P"}).fillna(df["option_type"].str.upper())
    )
    return df


def interpolate_iv(ticker, option_type, expiry, strike, chain_cache):
    """Interpolate market IV for a portfolio option from the downloaded chain."""
    chain = chain_cache.get(ticker.upper(), pd.DataFrame())
    if chain.empty:
        return np.nan, np.nan   # (iv, spot)

    # filter to same option type
    side = chain[chain["option_type"] == option_type.upper()]
    if side.empty:
        return np.nan, chain["underlying_price"].iloc[0] if "underlying_price" in chain.columns else np.nan

    # find closest expiry
    exp_dt = pd.to_datetime(expiry)
    unique_exps = pd.to_datetime(side["expiry"].unique())
    closest_exp = unique_exps[np.argmin(np.abs(unique_exps - exp_dt))]
    slice_df = side[side["expiry"] == closest_exp].sort_values("strike")

    if len(slice_df) < 2:
        return np.nan, np.nan

    iv_interp = float(np.interp(strike, slice_df["strike"].values, slice_df["implied_vol"].values))
    spot = float(slice_df["underlying_price"].iloc[0])
    return iv_interp, spot


# ── Build chain cache (one load per unique ticker) ────────────────────────────
tickers_needed = pf["symbol"].str.upper().unique()
chain_cache = {}
for tkr in tickers_needed:
    chain_cache[tkr] = _load_chain(tkr, DB_PATH, SNAPSHOT_DATE)
    snap_date = SNAPSHOT_DATE or (chain_cache[tkr]["snapshot_date"].max() if not chain_cache[tkr].empty else "N/A")
    n = len(chain_cache[tkr])
    print(f"  {tkr}: {n} chain rows  |  snapshot={snap_date}" if n else f"  {tkr}: NOT FOUND in DB — IV will be taken from CSV")

# ── Enrich portfolio ───────────────────────────────────────────────────────────
ivs, spots = [], []
for _, row in pf.iterrows():
    iv, spot = interpolate_iv(row["symbol"], row["option_type"], row["expiry"], row["strike"], chain_cache)
    ivs.append(iv)
    spots.append(spot)

pf["market_iv"] = ivs
pf["market_spot"] = spots

# Fall back to CSV values where market data is missing
pf["iv"] = np.where(np.isfinite(pf["market_iv"]), pf["market_iv"],
                    pf.get("implied_vol", pd.Series(np.nan, index=pf.index)))
pf["spot"] = np.where(np.isfinite(pf["market_spot"]), pf["market_spot"],
                      pf.get("spot", pd.Series(np.nan, index=pf.index)))

n_fresh = np.isfinite(pf["market_iv"]).sum()
print(f"\nIV interpolated from market data: {n_fresh}/{len(pf)} positions")
pf[["symbol","option_type","strike","expiry","iv","market_iv","spot"]].head(10)

In [ ]:
## 4 · Compute Greeks & probabilities on portfolio positions

# compute_option_metrics expects implied_vol column
pf_in = pf.rename(columns={"iv": "implied_vol", "option_price": "lastPrice",
                             "risk_free_rate": "r"})

metrics = compute_option_metrics(pf_in, default_rf=DEFAULT_RF, contract_size=CONTRACT_SIZE)

display_cols = [
    "symbol", "option_type", "strike", "expiry", "T", "implied_vol",
    "intrinsic", "extrinsic", "moneyness", "break_even", "max_loss",
    "prob_itm", "prob_profit", "prob_profit_ask",
    "delta", "gamma", "vega", "theta_day", "rho",
]
available = [c for c in display_cols if c in metrics.columns]
metrics[available]

In [ ]:
## 5 · Portfolio-level Greeks

port = portfolio_greeks(metrics)
pd.Series(port).rename("net_position").to_frame()

In [ ]:
## 6 · Probability chart — per position

labels = (
    metrics["symbol"] + " " + metrics["option_type"] +
    " K=" + metrics["strike"].astype(str) +
    " " + metrics["expiry"].dt.strftime("%b%y")
)

x = np.arange(len(metrics))
w = 0.28

fig, ax = plt.subplots(figsize=(max(10, len(metrics) * 1.4), 5))
ax.bar(x - w, metrics["prob_itm"],        width=w, label="P(ITM)",          color="steelblue")
ax.bar(x,     metrics["prob_profit"],      width=w, label="P(profit @ mid)", color="seagreen")
ax.bar(x + w, metrics["prob_profit_ask"],  width=w, label="P(profit @ ask)", color="orange")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylabel("Probability")
ax.set_title("Portfolio Positions — Probability Summary")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
## 7 · Portfolio payoff at expiry (all positions combined)

# Group by unique expiry and plot separate diagrams
unique_expiries = metrics["expiry"].dropna().unique()

for exp in sorted(unique_expiries):
    legs = metrics[metrics["expiry"] == exp]
    if legs.empty:
        continue

    spot_val = legs["spot"].iloc[0]
    S_range  = np.linspace(spot_val * 0.6, spot_val * 1.5, 500)
    total_pnl = np.zeros_like(S_range)

    for _, leg in legs.iterrows():
        K    = leg["strike"]
        prem = leg.get("lastPrice", 0) or 0
        qty  = leg.get("contracts", 1) * CONTRACT_SIZE
        if leg["option_type"] == "C":
            total_pnl += qty * (np.maximum(S_range - K, 0) - prem)
        else:
            total_pnl += qty * (np.maximum(K - S_range, 0) - prem)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(S_range, total_pnl, lw=2, color="navy")
    ax.axhline(0, color="black", lw=0.8)
    ax.axvline(spot_val, color="grey", lw=1, linestyle="--", alpha=0.7, label=f"Spot {spot_val:.1f}")
    ax.fill_between(S_range, total_pnl, 0, where=(total_pnl > 0), alpha=0.2, color="green", label="Profit")
    ax.fill_between(S_range, total_pnl, 0, where=(total_pnl < 0), alpha=0.2, color="red",   label="Loss")
    # Mark individual strikes
    for _, leg in legs.iterrows():
        ax.axvline(leg["strike"], color="purple", lw=0.8, linestyle=":", alpha=0.6)
    ax.set_xlabel("Underlying at expiry")
    ax.set_ylabel("P&L ($)")
    ax.set_title(f"Combined payoff at expiry {pd.Timestamp(exp).strftime('%d %b %Y')}")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
## 8 · Volatility smile — market chain with portfolio strikes marked

for tkr in tickers_needed:
    chain = chain_cache.get(tkr, pd.DataFrame())
    if chain.empty:
        print(f"No market chain for {tkr} — skipping smile chart.")
        continue

    # nearest expiry in the chain
    T_vals = chain["T"].dropna().unique() if "T" in chain.columns else []
    if len(T_vals) == 0:
        continue

    fig, ax = plt.subplots(figsize=(12, 5))
    cmap = plt.cm.viridis(np.linspace(0, 1, min(len(T_vals), 10)))

    for t_val, color in zip(sorted(T_vals)[:10], cmap):
        sub = chain[np.isclose(chain["T"], t_val, atol=1e-3)]
        calls = sub[sub["option_type"] == "C"].sort_values("strike")
        if len(calls) > 2:
            ax.plot(calls["strike"], calls["implied_vol"] * 100,
                    color=color, alpha=0.6, lw=1.5, label=f"T≈{t_val*365:.0f}d")

    # Mark portfolio strikes for this ticker
    pf_tkr = metrics[metrics["symbol"].str.upper() == tkr]
    for _, leg in pf_tkr.iterrows():
        ax.axvline(leg["strike"], color="red", lw=1.2, linestyle="--", alpha=0.8)
        ax.text(leg["strike"], ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 50,
                f'{leg["option_type"]}K{leg["strike"]:.0f}',
                rotation=90, fontsize=7, color="red", va="top")

    spot_val = chain["underlying_price"].iloc[0]
    ax.axvline(spot_val, color="black", lw=1.5, linestyle="--", alpha=0.5, label="Spot")
    ax.set_title(f"{tkr} — Implied Vol Smile  (red = portfolio strikes)")
    ax.set_xlabel("Strike")
    ax.set_ylabel("IV (%)")
    ax.legend(fontsize=7, ncol=4, loc="upper right")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()